In [ ]:
!pip install sympy matplotlib


In [ ]:
from sympy import randprime
import math
import time
import matplotlib.pyplot as plt

In [ ]:
def prime(n):
    if n < 2:
        return False

    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False

    return True

print(prime(23))

In [ ]:
import math
from sympy import randprime, nextprime, isprime
import random
import time
import matplotlib.pyplot as plt

def fermat_factor(n, max_iterations=2_000_000):
    """
    Attempts to factor n using Fermat's factorization method.
    Works by writing n = a^2 - b^2 = (a-b)(a+b), so if n = p*q,
    then p = a-b and q = a+b.

    Returns (p, q, iterations_used).
    If it can't find a factorization within max_iterations, returns (None, None, iterations_used).
    """
    a = math.isqrt(n)
    if a * a < n:
        a += 1  # a must be at least ceil(sqrt(n))

    iterations = 0
    while iterations < max_iterations:
        b_squared = a * a - n
        b = math.isqrt(b_squared)
        iterations += 1

        if b * b == b_squared:
            # found it: n = a^2 - b^2
            p, q = a - b, a + b
            return p, q, iterations

        a += 1  # try the next candidate

    return None, None, iterations

In [ ]:
def run_proximity_experiment(bit_size=32, max_distance=300, step=10, trials_per_distance=5):
    """
    For each distance d between p and q, generates several RSA-like
    n = p*q pairs and measures how many Fermat iterations it takes
    to factor n. Returns the distances and their average iteration counts.
    """
    distances = list(range(2, max_distance, step))
    avg_iterations = []

    for d in distances:
        total_iterations = 0
        successful_trials = 0

        for _ in range(trials_per_distance):
            p = randprime(2**(bit_size - 1), 2**bit_size)
            q = nextprime(p + d)  # q is roughly d away from p
            n = p * q

            _, _, iterations = fermat_factor(n, max_iterations=500_000)

            if iterations < 500_000:
                total_iterations += iterations
                successful_trials += 1

        if successful_trials > 0:
            avg_iterations.append(total_iterations / successful_trials)
        else:
            avg_iterations.append(None)  # couldn't factor within the limit

    return distances, avg_iterations

In [ ]:
distances, iterations = run_proximity_experiment(bit_size=32, max_distance=300)

plt.figure(figsize=(9, 5))
plt.plot(distances, iterations, marker='o')
plt.xlabel("Distance between p and q")
plt.ylabel("Average Fermat iterations")
plt.title("Factorization difficulty vs. p-q proximity")
plt.yscale('log')  # essential: growth is exponential-like, log scale reveals it clearly
plt.grid(True)
plt.show()

In [ ]:
bit_size = 40

# Case 1: p and q very close (max distance 3, like your original question)
p1 = randprime(2**(bit_size-1), 2**bit_size)
q1 = nextprime(p1 + 3)
n1 = p1 * q1
_, _, it1 = fermat_factor(n1)
print(f"Close primes (distance ~3): {it1} iterations")

# Case 2: p and q chosen independently at random (typical RSA)
p2 = randprime(2**(bit_size-1), 2**bit_size)
q2 = randprime(2**(bit_size-1), 2**bit_size)
n2 = p2 * q2
_, _, it2 = fermat_factor(n2, max_iterations=1_000_000)
print(f"Random primes: {it2}+ iterations (or timed out)")